In [63]:
from typing import List
import numpy as np
import os
from dotenv import load_dotenv

from qiskit import QuantumCircuit, generate_preset_pass_manager
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2

load_dotenv()

True

In [64]:
# Length of bitstream produced will be SHOTS * (NUM_QUBITS / PHYSICAL_PER_LOGICAL)
CHANNEL = 'local' # Channel to use for the runtime service. ibm_quantum or local
SHOTS = 10000 # How many times to run the circuit.
NUM_QUBITS = 9 # How many physical qubits to use. Must be a multiple of PHYSICAL_PER_LOGICAL.
PHYSICAL_PER_LOGICAL = 3 # How many physical qubits per logical qubit. Must be an odd number.
FILENAME = 'bitstream' # Filename to save the bitstream to.

assert NUM_QUBITS % PHYSICAL_PER_LOGICAL == 0
assert PHYSICAL_PER_LOGICAL % 2 == 1
assert (SHOTS * (NUM_QUBITS / PHYSICAL_PER_LOGICAL)) % 8 == 0 # Must be a multiple of 8 for the bitstream to be byte-aligned

In [65]:
# Defining Circuit
circ = QuantumCircuit(NUM_QUBITS)
circ.h(range(NUM_QUBITS))
circ.measure_all()

In [66]:
# Get backend and transpile circuit
service = QiskitRuntimeService(channel=CHANNEL, token=os.getenv('IBMQ_API_TOKEN'))
if CHANNEL == 'ibm_quantum':
    backend = service.least_busy(simulator=False, operational=True)
else:
    backend = service.least_busy()
pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circ = pm.run(circ)

In [67]:
# Initialise Sampler
sampler = SamplerV2(mode=backend)
sampler.options.default_shots = SHOTS

In [68]:
# Run the circuit
job = sampler.run([isa_circ])
print(f">>> Job ID: {job.job_id()}")

>>> Job ID: 5c19bf94-39c8-4bed-bf1c-73e228311feb


In [69]:
# Get the bitstreams
result = job.result()
bitstrings = np.array(result[0].data.meas.get_bitstrings())
bitstrings

array(['011001011', '001110111', '110011101', ..., '011010100',
       '100110110', '100001011'], shape=(10000,), dtype='<U9')

In [72]:
# Performa logical qubit majority vote
bitstream = []
for bitstring in bitstrings:
    bitstring = np.array([int(c) for c in bitstring]).reshape(-1, PHYSICAL_PER_LOGICAL)
    majority_votes = (np.mean(bitstring, axis=1) > 0.5).astype(int).tolist()
    bitstream.extend(majority_votes)
# bitstream = np.array(bitstream)
np.array(bitstream)

array([1, 0, 1, ..., 0, 0, 1], shape=(30000,))

In [71]:
# Write the bitstream to a file
bitstring = [str(bit) for bit in bitstream]
output = [bitstring[i:i+8] for i in range(0, len(bitstring), 8)]
ba = bytearray(int(''.join(byte), 2) for byte in output)
bs = bytes(ba)
with open(f"./{FILENAME}.bin", 'wb') as f:
    f.write(bs)